# Lab 12: Model Selection




What's today?

Even after chossing the hypothesis class, how do we choose the specific model that fits the best?
equivalent and more specific examples:
1. Choose k - number of neighbors in KNN algorithm
2. Choose depth of a tree in decision tree
3. Choose regularization factor 

The key question is, how to evaluate the generalization error (test error). We will go through few schemes:
1. Using the train set
2. Using validation set to evaluate the test error
3. Using cross validation 

In [1]:
import sys 
sys.path.append("../")
from utils import *

from scipy.stats import norm

from sklearn.model_selection import train_test_split, KFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

np.random.seed(7)

In [2]:
df = pd.read_csv("../data/SAheart.data", header=0, index_col=0).sort_values('chd')
df.famhist = df.famhist == "Present"

train, test = train_test_split(df, test_size=0.2)
X_train, y_train, X_test, y_test = train.loc[:, train.columns != 'chd'].values, train["chd"].values, test.loc[:, test.columns != 'chd'].values, test["chd"].values

FileNotFoundError: [Errno 2] No such file or directory: '../data/SAheart.data'

In [ ]:
X_train.shape

# First scheme

Choose the model that minimize the training error:

In [ ]:
k_range = range(1, 40, 2)

train_errors, test_errors = [], []

for k in k_range:
    model = KNeighborsClassifier(k)
    model.fit(X_train, y_train)
    train_errors.append(1 - model.score(X_train, y_train))
    test_errors.append(1 - model.score(X_test, y_test))

min_ind = np.argmin(np.array(train_errors))
selected_k = np.array(k_range)[min_ind]
selected_error = train_errors[min_ind]

go.Figure([go.Scatter(x=list(k_range), y=train_errors, mode='markers + lines', marker_color='rgb(152,171,150)', name=r'train errors'), 
          go.Scatter(x=list(k_range), y=test_errors, mode='markers + lines', marker_color='rgb(25,115,132)', name=r'test errors'),
          go.Scatter(x=[selected_k], y=[selected_error], mode='markers', marker_color='rgb(255, 0, 0)', name='selected model')])\
.update_layout(title="Errors of K-NN classifier", xaxis_title='k', yaxis_title='error').show()

Note how it looks (by the training error) that the model is best for k=1 (the most complex model).
It's clearly wrong, as k=1 doesn't perform the best, according to the test score.

Let's see if we can do better

# Second scheme

Single Validation set - 
1. Train the models on the training set
2. Evaluate the models on the evaluation set
3. Choose the model that performs the best on the evaluation set

Note:
That estimator of the generalization error is unbiased

In [ ]:
n = int(X_train.shape[0]*0.5)

X_train_smaller, y_train_smaller = X_train[:n], y_train[:n]
X_val, y_val = X_train[n:], y_train[n:]

train_errors, val_errors, test_errors = [], [], []

for k in k_range:
    model = KNeighborsClassifier(k)
    model.fit(X_train_smaller, y_train_smaller)
    train_errors.append(1 - model.score(X_train_smaller, y_train_smaller))
    val_errors.append(1 - model.score(X_val, y_val))
    test_errors.append(1-model.score(X_test, y_test))
    
min_ind = np.argmin(np.array(val_errors))
selected_k = np.array(k_range)[min_ind]
selected_error = val_errors[min_ind]

fig = go.Figure([ 
    go.Scatter(name='train error', x=list(k_range), y=train_errors, mode='markers+lines', marker_color='rgb(152,171,150)'),
    go.Scatter(name='validation error', x=list(k_range), y=val_errors, mode='markers+lines', marker_color='rgb(220,179,144)'),
    go.Scatter(name='test error', x=list(k_range), y=test_errors, mode='markers+lines', marker_color='rgb(25,115,132)'), 
    go.Scatter(x=[selected_k], y=[selected_error], mode='markers', marker_color='rgb(255, 0, 0)', name='selected model')
]).update_layout(title="Errors of knn classifier", xaxis_title="k", yaxis_title="error rate")
fig.show()

Now, instead of using a single validation, note what happens if we use few validation sets. Note how the validation sets results behave with respect to the train and test error.

The disadvantage here is that the train data is reduced one again. In the next sceme we will adress this issue.

In [ ]:
validate_sets = 3

n_train = X_train.shape[0]
n_small = np.int(np.floor(n_train * (1 - (validate_sets / 10 ))))
n_validate = np.int(np.floor(n_train * (validate_sets / 10 )) / validate_sets)


X_train_smaller, y_train_smaller = X_train[:n_small], y_train[:n_small]
cur_index = n_small
X_vals, y_vals = [], []
for val_set in range(validate_sets):
    X_vals.append(X_train[cur_index:cur_index+n_validate])
    y_vals.append(y_train[cur_index:cur_index+n_validate])
    cur_index += n_validate
train_errors, test_errors, vals_errors = [], [], [[] for _ in range(validate_sets)]


for k in k_range:
    model = KNeighborsClassifier(k)
    model.fit(X_train_smaller, y_train_smaller)
    train_errors.append(1-model.score(X_train_smaller, y_train_smaller))
    test_errors.append(1-model.score(X_test, y_test))

    for val_set in range(validate_sets): 
        vals_errors[val_set].append(1 - model.score(X_vals[val_set], y_vals[val_set]))

vals_errors = np.array(vals_errors)

mean, std = np.mean(vals_errors, axis=0), np.std(vals_errors, axis=0)
print(std)

single_validation = vals_errors[1, :]
min_ind = np.argmin(np.array(single_validation))
selected_k = np.array(k_range)[min_ind]
selected_error = single_validation[min_ind]



go.Figure([
    go.Scatter(name='Lower validation error', x=list(k_range), y=mean - 2*std, mode='lines', line=dict(color="lightgrey"), showlegend=False, fill=None),
    go.Scatter(name='Upper validation error', x=list(k_range), y=mean + 2*std, mode='lines', line=dict(color="lightgrey"), showlegend=False, fill="tonexty"), 

    go.Scatter(name='train error', x=list(k_range), y=train_errors, mode='markers+lines', marker_color='rgb(152,171,150)'),
    go.Scatter(name='validation error', x=list(k_range), y=single_validation, mode='markers+lines', marker_color='rgb(220,179,144)'),
    go.Scatter(name='test error', x=list(k_range), y=test_errors, mode='markers+lines', marker_color='rgb(25,115,132)'), 
    go.Scatter(x=[selected_k], y=[selected_error], mode='markers', marker_color='rgb(255, 0, 0)', name='selected model'), 

]).update_layout(title="Errors of knn classifier", xaxis_title="k", yaxis_title="error rate")\
.show()



# Third Scheme

cross validation explaination

In [ ]:
train_errors, test_errors = [], []

for k in k_range:
    model = KNeighborsClassifier(k)
    model.fit(X_train, y_train)
    train_errors.append(1-model.score(X_train, y_train))
    test_errors.append(1-model.score(X_test, y_test))

param_grid = {'n_neighbors':k_range}
knn_cv = GridSearchCV(KNeighborsClassifier(), param_grid, cv=3).fit(X_train, y_train)
cv_errors = 1 - knn_cv.cv_results_["mean_test_score"]
    
min_ind = np.argmin(np.array(cv_errors))
selected_k = np.array(k_range)[min_ind]
selected_error = cv_errors[min_ind]


go.Figure([go.Scatter(x=list(k_range), y=train_errors, mode='markers + lines', marker_color='rgb(152,171,150)', name=r'train error'), 
      go.Scatter(x=list(k_range), y=cv_errors, mode='markers + lines', marker_color='rgb(220,179,144)', name=r'CV error'),
      go.Scatter(x=list(k_range), y=test_errors, mode='markers + lines', marker_color='rgb(25,115,132)', name=r'test error'), 
      go.Scatter(x=[selected_k], y=[selected_error], mode='markers', marker_color='rgb(255, 0, 0)', name='selected model')])\
.update_layout(title="Scores of K-NN classifier", xaxis_title='k', yaxis_title='error').show()
